# Full Pipeline — Playground

End-to-end run of the **entire stock-selection pipeline** in one notebook, top to bottom.
Each stage below has a short explainer of *what* it does and *which dials* (in
`backend/config.py`) govern it.

```
Stage 1  Universe Filter   weekly   ~60-80 large-cap liquid stocks  → watchlist.csv
Stage 2  Momentum Scanner  nightly  scores the watchlist 0-3        → top candidates
Gate 1   Hard Threat       rules    blocks on macro/market shocks   (no Claude)
Gate 2   News Threat       Claude   blocks on catastrophic news
Gate 3   Sentiment         Claude   direction + confidence          → pass/block/caution
Gate 4   Contradiction     Claude   stock-vs-market divergence
Gate 5   Edge / EV         rules    win-probability + expected value → BUY / SKIP
Risk     Risk Gate         rules    sizing + portfolio limits         → approved / rejected
Exec     Alpaca Executor   paper    entry + trailing stop            → placed / skipped
```

The funnel narrows at every step: a few dozen universe names → a handful of momentum
candidates → the gates filter those down → the **Risk Gate** sizes and approves the final
**BUY** list → (opt-in) the **executor** places the orders.

> **⚠️ Live API + Claude calls.** Gates 2-4 each make one Claude (Haiku) call, so
> ~3 calls per surviving candidate. Needs `.env` keys: `ANTHROPIC_API_KEY` plus a news source
> (`ALPACA_API_KEY` + `ALPACA_SECRET_KEY`, or `NEWS_API_KEY`); the earnings/universe steps use
> `FINNHUB_API_KEY`. Missing keys degrade gracefully — gates **block** rather than crash.

> **⚠️ Portfolio source & paper orders are opt-in.**
> - `USE_LIVE_PORTFOLIO = False` → hardcoded defaults (dry-run sizing). Flip to `True` for
>   live Alpaca paper-account value / positions / P&L / drawdown.
> - `PLACE_ORDERS = False` → decide and size without trading. Flip to `True` only when you
>   want `position_trade()` to fire on each Risk-approved BUY (market must be open).

> **Every tunable number lives in `backend/config.py`** — the single dial board. This notebook
> only *reads* those values; tweak the strategy there.

## Setup — imports & path wiring

The numbered folders (`01_scanner`, `02_intelligence`, `03_risk`, `04_execution`) are **not**
importable Python packages, so — exactly like every other playground — we add the relevant
directories to `sys.path` and import each module by its bare name. We also add `backend/`
itself so `from config import ...` works here too.

Both `universe_filter` and `momentum_scanner` define a **cwd-relative** `WATCHLIST_PATH`
(`'data/watchlist.csv'`). We re-point both at the absolute path so the notebook runs no matter
where the kernel was launched.

In [8]:
import sys
import pathlib
import pandas as pd

# --- Anchor to backend/ regardless of where the kernel started -------------
nb_dir = pathlib.Path('.').resolve()
if (nb_dir / 'config.py').exists():
    backend_dir = nb_dir                                  # launched from backend/
else:
    backend_dir = pathlib.Path('backend').resolve()       # launched from repo root

intelligence_dir = backend_dir / '02_intelligence'
scanner_dir      = backend_dir / '01_scanner'
risk_dir         = backend_dir / '03_risk'
exec_dir         = backend_dir / '04_execution'

# Bare-name imports need each module's own dir on sys.path.
for p in [
    backend_dir,                                  # → config.py
    scanner_dir,                                  # → universe_filter, momentum_scanner
    intelligence_dir,                             # → constants, helpers/*
    risk_dir,                                     # → risk_gate
    exec_dir,                                     # → alpaca_executor
    intelligence_dir / 'gate1_hard_threat',
    intelligence_dir / 'gate2_news_threat',
    intelligence_dir / 'gate3_sentiment',
    intelligence_dir / 'gate4_contradiction',
    intelligence_dir / 'gate5_signal',
]:
    p = str(p)
    if p not in sys.path:
        sys.path.insert(0, p)

# --- Stage 1 & 2: scanner --------------------------------------------------
import universe_filter
import momentum_scanner
from universe_filter import run_universe_filter
from momentum_scanner import run_scan

# Re-point both cwd-relative watchlist paths at the real file.
WATCHLIST_PATH = str(scanner_dir / 'data' / 'watchlist.csv')
universe_filter.WATCHLIST_PATH = WATCHLIST_PATH
momentum_scanner.WATCHLIST_PATH = WATCHLIST_PATH

# --- Gates 1-5 -------------------------------------------------------------
from hard_threat_gate1 import get_shared_market_data, screen_gate1_hard_threats
from news_threat_gate2 import assess_gate2_news_threat
from sentiment_gate3 import evaluate_gate3_sentiment
from contradiction_gate4 import detect_gate4_contradiction
from signal_gate5 import decide_gate5_signal
from risk_gate import validate_trade

# --- Execution (Alpaca) ----------------------------------------------------
from alpaca_executor import (
    get_portfolio_value,
    get_open_positions,
    get_daily_pnl,
    get_drawdown_pct,
    position_trade,
)

# --- Fetchers shared across gates ------------------------------------------
from helpers.fetchers.news import fetch_news
from helpers.fetchers.market import get_market_context

# --- Active dials (read-only echo) -----------------------------------------
import config
print('Active strategy dials (edit in backend/config.py):')
print(f"  Universe : MIN_MARKET_CAP=${config.MIN_MARKET_CAP:,.0f}  MIN_PRICE=${config.MIN_PRICE}  "
      f"ATR%=[{config.MIN_ATR_PCT}, {config.MAX_ATR_PCT}]")
print(f"  Scanner  : MIN_SCORE={config.MIN_SCORE}  TOP_N={config.TOP_N}  "
      f"RSI=[{config.RSI_MIN}, {config.RSI_MAX}]")
print(f"  Gate 3   : MIN_CONFIDENCE={config.MIN_CONFIDENCE}")
print(f"  Gate 5   : MIN_EDGE_PCT={config.MIN_EDGE_PCT:.0%}  WIN_PROB_BASE={config.WIN_PROB_BASE:.0%}")
print(f"  Risk     : MAX_OPEN={config.MAX_OPEN_POSITIONS}  MAX_DAILY_LOSS={config.MAX_DAILY_LOSS_PCT:.0%}  "
      f"MAX_DRAWDOWN={config.MAX_DRAWDOWN_PCT:.0%}  KELLY={config.KELLY_FRACTION}")
print(f"  Watchlist: {WATCHLIST_PATH}")

Active strategy dials (edit in backend/config.py):
  Universe : MIN_MARKET_CAP=$100,000,000,000  MIN_PRICE=$10.0  ATR%=[1.0, 5.0]
  Scanner  : MIN_SCORE=2  TOP_N=15  RSI=[50, 70]
  Gate 3   : MIN_CONFIDENCE=6
  Gate 5   : MIN_EDGE_PCT=4%  WIN_PROB_BASE=35%
  Risk     : MAX_OPEN=5  MAX_DAILY_LOSS=3%  MAX_DRAWDOWN=8%  KELLY=0.25
  Watchlist: /Users/camilovargas/Documents/ai_bot/backend/01_scanner/data/watchlist.csv


## Stage 1 — Universe Filter  *(weekly screen → `watchlist.csv`)*

The Tier-1 screen defines **which stocks are even allowed onto the watchlist**. It asks
TradingView for US large-caps and keeps only the liquid, tradable ones, then drops names with
earnings in the next few days. The survivors (with `price, volume, atr, rsi, sma20, sma50,
sector`) are written to `watchlist.csv`.

Dials (`config.py`): `MIN_MARKET_CAP` (≥ \$100B), `MIN_AVG_VOLUME` (≥ 1M shares/day),
`MIN_PRICE` (≥ \$10), `MIN_ATR_PCT`/`MAX_ATR_PCT` (1–5% daily range), `EARNINGS_WINDOW_DAYS`
(exclude names reporting within 5 days), `SCREENER_LIMIT`.

This is a **weekly** job and a live regeneration overwrites the existing watchlist (TradingView
+ Finnhub calls). So it is **opt-in**: by default we just read the current `watchlist.csv` and
report what's in it. Flip `RUN_UNIVERSE_FILTER = True` to rebuild it live.

In [9]:
RUN_UNIVERSE_FILTER = True   # set True to rebuild watchlist.csv live (slow; overwrites!)

if RUN_UNIVERSE_FILTER:
    # max_price passed explicitly to skip the Alpaca portfolio lookup (reproducible runs).
    count = run_universe_filter(max_price=400.0)
    print(f'[universe] regenerated watchlist with {count} stocks')

universe_df = pd.read_csv(WATCHLIST_PATH)
print(f'Universe: {len(universe_df)} stocks in watchlist.csv\n')

# Sector breakdown — how the universe is spread across the market.
print('By sector:')
print(universe_df['sector'].value_counts().to_string())

print('\nSample rows:')
universe_df.head(10)

[universe] price ceiling: $400.00
[universe] querying TradingView screener...
[universe] 74 stocks after ATR% filter (1.0%–5.0%)
[universe] 12 stocks removed for upcoming earnings
[universe] saved 62 tickers to /Users/camilovargas/Documents/ai_bot/backend/01_scanner/data/watchlist.csv
[universe] regenerated watchlist with 62 stocks
Universe: 62 stocks in watchlist.csv

By sector:
sector
Electronic Technology    11
Health Technology         8
Technology Services       8
Finance                   8
Retail Trade              6
Consumer Non-Durables     5
Consumer Services         4
Communications            3
Energy Minerals           3
Transportation            2
Utilities                 2
Consumer Durables         1
Non-Energy Minerals       1

Sample rows:


,ticker,price,volume,atr,atr_pct,rsi,sma20,sma50,sector
0,NVDA,203.53,138567293,7.22,3.55,49.9,201.88,209.08,Electronic Technology
1,T,21.55,89313829,0.67,3.11,42.5,21.85,23.50,Communications
2,AAPL,317.31,51994751,8.09,2.55,64.2,299.19,298.71,Electronic Technology
3,AMZN,247.31,46057352,7.56,3.06,52.9,240.48,253.72,Retail Trade
4,PFE,24.48,45039854,0.54,2.22,46.1,24.63,25.40,Health Technology
5,TSLA,394.76,44574610,18.59,4.71,47.2,400.66,409.64,Consumer Durables
6,VZ,42.68,42582926,1.29,3.02,38.6,44.43,46.13,Communications
7,MSFT,390.99,36043033,11.80,3.02,51.0,380.55,402.95,Technology Services
8,AVGO,384.05,24749235,17.44,4.54,48.1,382.69,406.02,Electronic Technology
9,CSCO,119.25,23410441,3.90,3.27,54.2,117.81,114.35,Electronic Technology


## Stage 2 — Momentum Scanner  *(nightly → top candidates)*

The Tier-2 scan scores every watchlist stock **0–3**, one point each for:

1. RSI inside the momentum zone `[RSI_MIN, RSI_MAX]` — rising but not yet overbought,
2. price **>** SMA20 — short-term uptrend,
3. SMA20 **>** SMA50 — bullish structure (golden-cross alignment).

It keeps names scoring ≥ `MIN_SCORE` and returns the strongest `TOP_N` by score.

Dials (`config.py`): `MIN_SCORE` (=2), `TOP_N` (=15), `RSI_MIN`/`RSI_MAX` (50–70).

`run_scan()` includes the `sector` column directly (Gates 1 & 4 both need `candidate['sector']`).

In [11]:
candidates_df = run_scan()   # uses MIN_SCORE / TOP_N from config

if candidates_df is None or candidates_df.empty:
    print('No candidates from run_scan — check watchlist.csv')
else:
    print(f'\nFunnel so far: {len(universe_df)} universe '
          f'→ {len(candidates_df)} candidates (score ≥ {config.MIN_SCORE}, top {config.TOP_N})')

candidates_df

[scanner] loaded 62 stocks from /Users/camilovargas/Documents/ai_bot/backend/01_scanner/data/watchlist.csv
[scanner] 43 stocks with score >= 2
[scanner] returning top 15 candidates

Funnel so far: 62 universe → 15 candidates (score ≥ 2, top 15)


,ticker,price,score,atr,rsi,sma20,sma50,sector
0,AAPL,317.31,3,8.09,64.2,299.19,298.71,Electronic Technology
1,COF,203.02,3,5.88,56.5,200.45,191.68,Finance
2,MO,71.87,3,1.65,50.8,71.66,71.65,Consumer Non-Durables
3,FTNT,160.62,3,6.57,62.4,152.77,135.76,Technology Services
4,SO,96.47,3,1.70,55.2,95.37,94.10,Utilities
5,IBKR,93.56,3,3.74,55.0,93.04,88.16,Finance
6,RTX,196.39,3,4.43,60.6,190.64,182.33,Electronic Technology
7,BX,122.04,3,4.12,53.4,120.75,119.84,Finance
8,DHR,200.16,3,5.22,64.5,189.08,180.59,Health Technology
9,IBM,290.23,3,12.25,56.7,277.45,262.90,Technology Services


## Run-level inputs

Before the gate loop we set the **portfolio context** Gate 1 and the Risk Gate need.
`USE_LIVE_PORTFOLIO` picks the source:

- `True`  → live paper account via Alpaca (`get_portfolio_value`, `get_open_positions`,
  `get_daily_pnl`, `get_drawdown_pct`). If any call fails, we fall back to the hardcoded
  defaults so decisioning still runs.
- `False` → the hardcoded defaults below (reproducible dry-runs, no Alpaca dependency).

We also fetch the **market-wide data once** (`get_shared_market_data()` → VIX / SPY /
hours-to-next-macro) so every Gate 1 call reuses it.

`TOP_N` caps how many candidates we push through the (paid) Claude gates.
`PLACE_ORDERS` is the paper-order safety switch — leave it `False` unless you mean to trade.

In [17]:
# Portfolio defaults — used when USE_LIVE_PORTFOLIO is False, or as Alpaca fallback.
portfolio_value        = 100_000.0
daily_pnl              = 0.0
open_positions_count   = 0
drawdown_pct           = 0.0
TOP_N                  = 10           # cap candidates pushed through the Claude gates
USE_LIVE_PORTFOLIO     = True        # True → Alpaca paper account; False → defaults above
PLACE_ORDERS           = True        # True → position_trade() on each Risk-approved BUY

if USE_LIVE_PORTFOLIO:
    live_pv = get_portfolio_value()
    live_positions = get_open_positions()
    live_pnl = get_daily_pnl()
    live_dd = get_drawdown_pct()

    if live_pv is not None:
        portfolio_value = live_pv
    if live_positions is not None:
        open_positions_count = len(live_positions)
    if live_pnl is not None:
        daily_pnl = live_pnl
    if live_dd is not None:
        drawdown_pct = live_dd

    source = 'Alpaca' if live_pv is not None else 'fallback defaults (Alpaca unavailable)'
else:
    source = 'defaults'

shared = get_shared_market_data()   # VIX / SPY / macro — fetched once for all Gate 1 calls
print('Shared market data:', shared)
print(f'Portfolio ({source}): ${portfolio_value:,.2f}  daily_pnl=${daily_pnl:+,.2f}  '
      f'open_positions={open_positions_count}  drawdown={drawdown_pct:.2%}')
print(f'USE_LIVE_PORTFOLIO={USE_LIVE_PORTFOLIO}  PLACE_ORDERS={PLACE_ORDERS}')

Shared market data: {'vix': {'level': 17.16, 'change_pct_today': 0.1417, 'prior_close': 15.03}, 'spy': {'price': 749.17, 'change_pct_today': -0.0077, 'prior_close': 754.95}, 'macro_hours': 14.4}
Portfolio (Alpaca): $100,000.58  daily_pnl=$+0.58  open_positions=0  drawdown=0.00%
USE_LIVE_PORTFOLIO=True  PLACE_ORDERS=True


## Gates 1–5, Risk & Execution — the decision chain

Each candidate runs the gates **in order, stopping at the first failure** (fail fast keeps
Claude cost down). The chain:

- **Gate 1 — Hard Threat** *(rules)*: blocks on macro/market shocks — VIX spike, SPY/sector
  selloff, pre-market gap, imminent macro event, earnings tomorrow, fresh 8-K, daily loss
  limit. Thresholds in `config.BLOCK_THRESHOLDS`.
- **Gate 2 — News Threat** *(Claude)*: reads the headlines, blocks on a catastrophic story
  (fraud, recall, regulatory action, …). News is fetched **once** here and reused by Gate 3.
- **Gate 3 — Sentiment** *(Claude)*: returns direction + confidence (0–10). `MIN_CONFIDENCE`
  turns that into pass / block / pass-with-caution.
- **Gate 4 — Contradiction** *(Claude)*: blocks on HIGH-risk contradictions; LOW/MEDIUM flags
  pause the run and prompt **Y/N** — Y continues to Gate 5, N skips the ticker.
- **Gate 5 — Edge / EV** *(rules)*: maps momentum score + Gate 3 sentiment to a win
  probability, computes expected value, and issues **BUY** if `EV ≥ MIN_EDGE_PCT` else
  **SKIP**. Also returns the trade levels (entry/stop/target/`stop_pct`).
- **Risk Gate** *(rules, zero Claude cost)*: last-mile check on every Gate 5 **BUY** — open
  position count, drawdown kill switch, daily loss limit, reward:risk floor, and Quarter-Kelly
  position sizing.
- **Execution** *(Alpaca, opt-in)*: when `PLACE_ORDERS` is `True`, each Risk-approved BUY
  calls `position_trade()` — market entry, then a native trailing stop whose `trail_percent`
  comes from Gate 5's `stop_pct`. When `False`, the row stays `BUY` (sized, not placed).

The driver below records, for every ticker, where it stopped and the Gate 3 / Gate 5 / Risk /
exec numbers so the results table tells the whole story.

In [18]:
def _confirm_flag(ticker, g4):
    """Ask human Y/N when Gate 4 flags a contradiction for review."""
    print(f'\n[review] {ticker} flagged — {g4["contradiction_type"]} risk={g4["risk_level"]}')
    print(f'         {g4["reason"]}')
    while True:
        answer = input(f'Proceed with {ticker}? [Y/N]: ').strip().upper()
        if answer == 'Y':
            return True
        if answer == 'N':
            return False
        print('Please enter Y or N.')

def _row(ticker, decision, g3=None, g5=None, risk=None, exec_audit=None):
    """One results-table row; gate3/gate5/risk/exec fields filled only when those stages ran."""
    tl = g5.get('trade_levels') if g5 else None
    pos = risk['position'] if risk else None
    return {
        'ticker': ticker,
        'final_decision': decision,
        'g3_direction': g3.get('direction') if g3 else None,
        'g3_confidence': g3.get('confidence') if g3 else None,
        'ev': round(g5['expected_value'], 3) if g5 else None,
        'win_prob': round(g5['win_probability'], 3) if g5 else None,
        'position_confidence': g5['position_confidence'] if g5 else None,
        'entry': tl['entry'] if tl else None,
        'stop': tl['stop'] if tl else None,
        'target': tl['target'] if tl else None,
        'reward_risk': tl['reward_risk'] if tl else None,
        'shares': pos['shares'] if pos else None,
        'position_value': pos['position_value'] if pos else None,
        'position_pct': pos['position_pct'] if pos else None,
        'risk_reject': risk.get('reject_reason') if risk and not risk['approved'] else None,
        'filled_qty': exec_audit.get('filled_qty') if exec_audit else None,
        'filled_avg_price': exec_audit.get('filled_avg_price') if exec_audit else None,
        'trail_percent': exec_audit.get('trail_percent') if exec_audit else None,
        'stop_attached': exec_audit.get('stop_attached') if exec_audit else None,
    }

rows, processed = [], 0

if candidates_df is not None and not candidates_df.empty:
    for _, r in candidates_df.iterrows():
        if processed >= TOP_N:
            break
        ticker = str(r.at['ticker'])
        sector_val = r.at['sector']
        if not isinstance(sector_val, str):
            print(f'{ticker:<5} skipped — no sector in watchlist')
            continue
        sector = sector_val
        processed += 1

        candidate = {
            'ticker': ticker,
            'sector': sector,
            'price': float(r['price']),
            'atr': float(r['atr']),
            'score': int(r['score']),
        }

        # Gate 1 — hard threats (rules, reuses shared market data)
        g1 = screen_gate1_hard_threats(candidate, shared, portfolio_value, daily_pnl)
        if not g1['passed']:
            rows.append(_row(ticker, f"BLOCKED_G1:{g1.get('block_reason')}"))
            continue

        # Gates 2 & 3 share a single news fetch
        headlines = fetch_news(ticker) or []
        g2 = assess_gate2_news_threat(candidate, headlines)
        if not g2['passed']:
            rows.append(_row(ticker, 'BLOCKED_G2'))
            continue
        g3 = evaluate_gate3_sentiment(candidate, headlines)
        if not g3['passed']:
            rows.append(_row(ticker, 'BLOCKED_G3', g3=g3))
            continue

        # Gate 4 — contradiction vs the live market backdrop
        market_context = get_market_context(sector)
        if market_context is None:
            rows.append(_row(ticker, 'BLOCKED_G4:no_market_context', g3=g3))
            continue
        g4 = detect_gate4_contradiction(candidate, g3, market_context)
        if g4['action'] == 'BLOCK':
            rows.append(_row(ticker, 'BLOCKED_G4', g3=g3))
            continue
        if g4['action'] == 'FLAG_FOR_REVIEW':
            if not _confirm_flag(ticker, g4):
                rows.append(_row(ticker, 'REJECTED_FLAG', g3=g3))
                continue

        # Gate 5 — edge check + EV
        g5 = decide_gate5_signal(candidate, {'gate1': g1, 'gate2': g2, 'gate3': g3, 'gate4': g4})
        if g5['decision'] != 'BUY':
            rows.append(_row(ticker, g5['decision'], g3=g3, g5=g5))
            continue

        # Risk gate — sizing + portfolio limits (rules only, zero Claude cost)
        risk = validate_trade(ticker, g5, portfolio_value, daily_pnl,
                              open_positions_count, drawdown_pct)
        if not risk['approved']:
            rows.append(_row(ticker, f"REJECTED_RISK:{risk['reject_reason']}", g3=g3, g5=g5, risk=risk))
            continue

        # Execution — opt-in paper orders (entry + trailing stop)
        if not PLACE_ORDERS:
            rows.append(_row(ticker, 'BUY', g3=g3, g5=g5, risk=risk))
            continue

        audit = position_trade({
            'ticker': ticker,
            'shares': risk['position']['shares'],
            'trade_levels': g5['trade_levels'],
        })
        if audit is None:
            rows.append(_row(ticker, 'EXEC_FAILED', g3=g3, g5=g5, risk=risk))
            continue

        open_positions_count += 1  # keep Risk Gate's open-count check honest mid-run
        decision = 'PLACED' if audit['stop_attached'] else 'PLACED_UNPROTECTED'
        rows.append(_row(ticker, decision, g3=g3, g5=g5, risk=risk, exec_audit=audit))

print(f'\nProcessed {processed} candidates through the gates.')
print(f'PLACE_ORDERS={PLACE_ORDERS}')

[gate1] AAPL: BLOCKED — sector
[gate1] COF: passed all 8 checks
[gate2] COF: passed — no threat across 5 headlines
[gate3] COF: passed — BULLISH conf=9
[gate4] COF: FLAG_FOR_REVIEW — divergence risk=MEDIUM: COF is in a sector (XLF +0.65%) that is outperforming a weakening broad market (SPY -0.77%), creating relative strength that may snap back as risk-off sentiment deepens given the sharp VIX spike (+14.17%).

[review] COF flagged — divergence risk=MEDIUM
         COF is in a sector (XLF +0.65%) that is outperforming a weakening broad market (SPY -0.77%), creating relative strength that may snap back as risk-off sentiment deepens given the sharp VIX spike (+14.17%).
[gate5] COF: BUY — EV 1.062 | win_prob=69% | HIGH
[risk] COF: APPROVED — 39 shares ($8,000.05, 8.0% of portfolio)


KeyboardInterrupt: 

## Results — the funnel, approved BUYs & placements

The table below shows every processed ticker, where it stopped, and the Gate 3 / Gate 5 / Risk /
exec numbers. Then the funnel counts
(`universe → scanned → processed → Gate-5 BUY → approved → placed`) and, for each **approved
BUY**, the trade levels and Quarter-Kelly size. When `PLACE_ORDERS` was on, placed rows also
show fill qty / avg price / trail % / stop-attached.

In [16]:
results_df = pd.DataFrame(rows)

if results_df.empty:
    print('No candidates were processed.')
else:
    print(results_df.to_string(index=False))

    approved = results_df[results_df['final_decision'].isin(
        ['BUY', 'PLACED', 'PLACED_UNPROTECTED']
    )]
    placed = results_df[results_df['final_decision'].isin(['PLACED', 'PLACED_UNPROTECTED'])]
    exec_failed = results_df[results_df['final_decision'] == 'EXEC_FAILED']
    gate5_buys = results_df[
        results_df['final_decision'].isin(['BUY', 'PLACED', 'PLACED_UNPROTECTED', 'EXEC_FAILED'])
        | results_df['final_decision'].str.startswith('REJECTED_RISK', na=False)
    ]
    risk_rejected = results_df[results_df['final_decision'].str.startswith('REJECTED_RISK', na=False)]
    scan_df = candidates_df if candidates_df is not None else pd.DataFrame()
    print(f'\nFunnel: {len(universe_df)} universe '
          f'→ {len(scan_df)} scanned '
          f'→ {processed} processed '
          f'→ {len(gate5_buys)} Gate-5 BUY '
          f'→ {len(approved)} approved '
          f'→ {len(placed)} placed')

    if not approved.empty:
        print('\nApproved trades (Risk Gate):')
        for _, b in approved.iterrows():
            print(f"  {b['ticker']:<5} {int(b['shares'])} shares (${b['position_value']:,.2f}, "
                  f"{b['position_pct']:.1%})  "
                  f"entry={b['entry']:.2f}  stop={b['stop']:.2f}  target={b['target']:.2f}  "
                  f"R:R={b['reward_risk']:.1f}  EV={b['ev']}  conf={b['position_confidence']}  "
                  f"→ {b['final_decision']}")

    if not placed.empty:
        print('\nPlaced orders (Alpaca):')
        for _, p in placed.iterrows():
            print(f"  {p['ticker']:<5} filled={int(p['filled_qty'])} @ ${p['filled_avg_price']:,.2f}  "
                  f"trail={p['trail_percent']}%  stop_attached={p['stop_attached']}")

    if not exec_failed.empty:
        print('\nExecution failed (no position opened):')
        for _, e in exec_failed.iterrows():
            print(f"  {e['ticker']:<5} shares={int(e['shares'])}  EV={e['ev']}")

    if not risk_rejected.empty:
        print('\nGate-5 BUY rejected by Risk Gate:')
        for _, r in risk_rejected.iterrows():
            print(f"  {r['ticker']:<5} reason={r['risk_reject']}  EV={r['ev']}")

ticker    final_decision g3_direction  g3_confidence    ev  win_prob position_confidence  entry    stop  target  reward_risk  shares  position_value  position_pct risk_reject filled_qty filled_avg_price trail_percent stop_attached
  AAPL BLOCKED_G1:sector         None            NaN   NaN       NaN                None    NaN     NaN     NaN          NaN     NaN             NaN           NaN        None       None             None          None          None
   COF               BUY      BULLISH            9.0 1.062     0.688                HIGH 203.02 194.200  220.66          2.0    39.0         8000.05        0.0800        None       None             None          None          None
    MO               BUY      BULLISH            7.0 0.688     0.562                HIGH  71.87  69.395   76.82          2.0   111.0         8000.05        0.0800        None       None             None          None          None
  FTNT BLOCKED_G1:sector         None            NaN   NaN       NaN        

## Free-play

Scratch cell — tweak `USE_LIVE_PORTFOLIO`, `portfolio_value`, `open_positions_count`,
`drawdown_pct`, `TOP_N`, `PLACE_ORDERS`, or a single candidate and re-run pieces above.